In [36]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [39]:
batch_size = 16
IMG_SIZE = (224, 224)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/train",
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/val",
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode="categorical"
)


Found 4835 images belonging to 3 classes.
Found 1031 images belonging to 3 classes.


In [4]:
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


In [5]:
for layer in base_model.layers[-80:]:
    layer.trainable = True



In [6]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

callbacks = [
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )
]


In [7]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
Dense(512, activation="relu"),
Dropout(0.6),
Dense(256, activation="relu"),
Dropout(0.5)

x = Dropout(0.5)(x)
output = Dense(3, activation="softmax")(x)  # covid, normal, pneumonia

model = Model(inputs=base_model.input, outputs=output)


In [8]:
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [10]:
history = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen
)


Epoch 1/15


ValueError: Creating variables on a non-first call to a function decorated with tf.function.

In [41]:
from tensorflow.keras.losses import CategoricalCrossentropy

model.compile(
    optimizer=Adam(1e-5),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)



In [21]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


{0: np.float64(1.0091838864537674), 1: np.float64(0.9954704550133827), 2: np.float64(0.9954704550133827)}


In [22]:
model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    callbacks=callbacks
)



Epoch 1/30
 24/303 ━━━━━━━━━━━━━━━━━━━━ 11:50 3s/step - accuracy: 0.3284 - loss: 1.8274

KeyboardInterrupt: 

In [23]:
test_gen = val_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/test",
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)
model.evaluate(test_gen)


Found 1036 images belonging to 3 classes.
33/65 ━━━━━━━━━━━━━━━━━━━━ 13s 424ms/step - accuracy: 0.1078 - loss: 2.0268

KeyboardInterrupt: 

In [40]:
import tensorflow as tf
tf.keras.backend.clear_session()


In [20]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


In [42]:
for layer in base_model.layers:
    layer.trainable = False


In [45]:
from tensorflow.keras.layers import BatchNormalization

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(512, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(256, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

output = Dense(3, activation="softmax")(x)
model = Model(base_model.input, output)



In [46]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [47]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

callbacks = [
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        "best_model.h5",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]


In [ ]:
model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks   # ✅ AGAIN HERE
)


Epoch 1/10
 56/303 ━━━━━━━━━━━━━━━━━━━━ 1:54 464ms/step - accuracy: 0.4299 - loss: 1.4407

In [30]:
for layer in base_model.layers:
    layer.trainable = False

for layer in base_model.layers[-40:]:  # last dense block
    layer.trainable = True


In [34]:
loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)


In [35]:
model.compile(
    optimizer=Adam(1e-5),
    loss=loss,
    metrics=["accuracy"]
)



In [33]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


{0: np.float64(1.0091838864537674), 1: np.float64(0.9954704550133827), 2: np.float64(0.9954704550133827)}


In [29]:
model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks,
    class_weight=class_weights
)


Epoch 1/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 560ms/step - accuracy: 0.6418 - loss: 1.3679
Epoch 1: val_accuracy did not improve from 0.88749
303/303 ━━━━━━━━━━━━━━━━━━━━ 211s 666ms/step - accuracy: 0.7421 - loss: 0.8548 - val_accuracy: 0.8749 - val_loss: 0.3412 - learning_rate: 1.0000e-05
Epoch 2/15
211/303 ━━━━━━━━━━━━━━━━━━━━ 1:01 667ms/step - accuracy: 0.8350 - loss: 0.4789

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes

print(classification_report(y_true, y_pred, target_names=test_gen.class_indices.keys()))
